# 위밋모빌리티 — 해상 리스크 대응형 공동 물류 운영 플랫폼 (v4)

> **부제**: 중소 수출기업의 해상 차질 발생 시 출하 수요를 운영 단위로 재조정하고,
> 위밋 루티/루티프로 연계용 실행 조건을 생성하는 의사결정 서비스

---

## 시스템 구성

**Part 1 — 리스크 모니터링 & 평상시 운영** (v3 기반)
1. 환경 설정
2. 뉴스 수집 (RSS/시뮬)
3. NLP 리스크 분류 (키워드 사전)
4. MRI 산출 (KCCI 실데이터 옵션)
5. 부산항 물동량 + 거시경제 (ECOS API 옵션)
6. LSTM 물동량 예측
7. 공동선적 매칭 (평상시 배차)

**Part 2 — 시나리오 기반 운영 재조정** (NEW)
8. 시나리오 정의 (A/B/C/D/E + 자동 분류기)
9. 출하 예정 건 등록
10. 영향 분석 엔진
11. 운영 재조정 엔진
12. 루티 연계 JSON 출력

**Part 3 — 통합 시연**
13. 5개 시나리오 일괄 실행 비교
14. 핵심 KPI 대시보드
15. 발표용 케이스 스터디 (경기남부 A/B/C사)

---

## v3 → v4 변경점

| 영역 | v3 | v4 |
|---|---|---|
| 핵심 컨셉 | 공동선적 + ESG | 해상 리스크 대응 운영 재조정 |
| 매칭 알고리즘 | 그리디 + 다목적 점수 | + 시나리오별 정책 분기 |
| 출력 | 매칭 그룹 표 | 루티 API 입력용 JSON 표준 |
| 시나리오 | 단일 (평상시) | 5종 (정상/지정학/기상/지연/취소) |

---

## 실행 환경

- **권장**: 로컬 Jupyter Notebook 또는 VSCode (Anaconda)
- Python 3.10+
- 메모리: 4GB 이상
- 인터넷: 선택사항 (실데이터 API 사용 시)

> Colab에서도 실행 가능하나, 본 노트북은 로컬 환경 기준으로 설계됨

---
# Part 1 — 리스크 모니터링 & 평상시 운영

v3 정리된 코드 기반. 뉴스 수집 → NLP 분류 → MRI 산출 → 부산항 물동량 →
LSTM 예측 → 공동선적 매칭까지 평상시 운영 흐름.

## 1. 환경 설정

In [ ]:
# ── 필요 라이브러리 설치 (이미 설치되어 있으면 건너뜀) ──
# 주석 해제하여 사용
# !pip install pandas numpy matplotlib seaborn plotly scikit-learn xgboost torch requests beautifulsoup4

import os, warnings, json, random
from pathlib import Path
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# Plotly (시각화)
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    PLOTLY_OK = True
except ImportError:
    print('Plotly 미설치 — pip install plotly 후 재실행')
    PLOTLY_OK = False

# PyTorch (LSTM)
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    TORCH_OK = True
except ImportError:
    print('PyTorch 미설치 — Part 1 LSTM은 건너뜀. pip install torch')
    TORCH_OK = False

# scikit-learn / XGBoost
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error, classification_report, accuracy_score
import xgboost as xgb

# 시드 고정
np.random.seed(42)
random.seed(42)

# 한글 폰트 자동 설정
def setup_kr_font():
    candidates = ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'NanumBarunGothic']
    available = {f.name for f in fm.fontManager.ttflist}
    for c in candidates:
        if c in available:
            plt.rcParams['font.family'] = c
            return c
    print('⚠️ 한글 폰트 없음 — 기본 폰트 사용')
    return None

KR_FONT = setup_kr_font()
plt.rcParams['axes.unicode_minus'] = False

# 데이터 폴더 (실데이터 사용 시)
DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)

# 실데이터 토글 (False면 시뮬레이션)
USE_REAL_KCCI       = False  # data/kcci.csv 있으면 True로
USE_REAL_THROUGHPUT = False  # data/busan_throughput.csv 있으면 True로
USE_REAL_MACRO      = False  # ECOS API 키 입력 후 True로
ECOS_API_KEY = "여기에_ECOS_API_키_입력"

print(f'한글 폰트: {KR_FONT}')
print(f'PyTorch: {"OK" if TORCH_OK else "미설치"}, Plotly: {"OK" if PLOTLY_OK else "미설치"}')
print(f'데이터 폴더: {DATA_DIR.resolve()}')

## 2. 뉴스 수집

본 단계는 시뮬레이션 데이터 기반. 실서비스 시 RSS/API 연동으로 확장 가능.

In [ ]:
# 시뮬레이션 뉴스 (실제 해운 뉴스 패턴 반영)
news_simulated = [
    {'title': '홍해 후티 반군 공격 재개, 컨테이너 운임 급등', 'text': '홍해 항로 봉쇄로 운임 30% 상승 우려', 'pub_date': '2026-04-28'},
    {'title': '부산항 컨테이너 처리량 사상 최대 갱신', 'text': '4월 처리량 전년 대비 4.8% 증가', 'pub_date': '2026-04-25'},
    {'title': '태풍 카눈 북상, 부산항 입출항 차질 우려', 'text': '기상청 태풍 경보, 컨테이너선 피항 준비', 'pub_date': '2026-04-26'},
    {'title': '미중 추가 관세 부과, 환적 물동량 감소', 'text': '관세 정책 변동으로 수출 기업 타격', 'pub_date': '2026-04-22'},
    {'title': '호르무즈 해협 긴장 고조, 이란 미사일 위협', 'text': '지정학적 분쟁 위험 확대', 'pub_date': '2026-04-20'},
    {'title': '부산항 노조 부분 파업, 하역 지연', 'text': '항만 파업으로 컨테이너 체류 시간 증가', 'pub_date': '2026-04-18'},
    {'title': '글로벌 선사 운임 인상 발표', 'text': 'Maersk, MSC 운임 5월부터 상승', 'pub_date': '2026-04-15'},
    {'title': '한국 조선업 LNG선 수주 회복', 'text': '글로벌 LNG선 발주 증가', 'pub_date': '2026-04-10'},
    {'title': 'EU CBAM 본격 시행 임박, 수출기업 비상', 'text': 'CBAM 인증서 구매 부담 가중', 'pub_date': '2026-04-08'},
    {'title': '부산항 신항 물동량 회복세', 'text': '신항 정상화로 처리능력 향상', 'pub_date': '2026-04-05'},
]
news_df = pd.DataFrame(news_simulated)
print(f'뉴스 수집 완료: {len(news_df)}건')
news_df.head()

## 3. NLP 리스크 분류

해운 도메인 키워드 사전 기반 다중 카테고리 분류 + 규칙 기반 감성 판정.
향후 KoBERT 임베딩 적용 시 정밀도·재현율 80% 이상 목표.

In [ ]:
# ── 리스크 카테고리 키워드 사전 ──
RISK_KEYWORDS = {
    '지정학분쟁': ['봉쇄', '전쟁', '분쟁', '공격', '테러', '후티', '이란', '러시아',
                   '미사일', '충돌', '긴장', '갈등', '적대', '위협', '호르무즈', '홍해'],
    '항만파업':   ['파업', '파멸', '노조', '총파업', '시위', '불법', '거부',
                   '협상', '체증', '하역', '부두'],
    '기상재해':   ['태풍', '폭풍', '가뭄', '홍수', '지진', '해일', '강풍', '기상',
                   '이상기후', '운하', '수위', '결항'],
    '관세정책':   ['관세', '제재', '규제', '무역전쟁', 'IMO', '환경규제', '정책',
                   '금지', '추가관세', '관세부과', 'CBAM'],
    '운임급등':   ['운임', '급등', '상승', 'SCFI', 'BDI', 'KCCI', '운임지수', '선복',
                   '부족', '비용', '증가', '인상'],
    '정상':       ['정상화', '완화', '회복', '협력', '개설', '투자',
                   '성장', '최대', '안정'],
}
RISK_WEIGHTS = {
    '지정학분쟁': 1.0, '항만파업': 0.85, '기상재해': 0.75,
    '관세정책': 0.65, '운임급등': 0.55, '정상': 0.0
}

def classify_risk(text: str) -> dict:
    scores = {cat: sum(1 for kw in kws if kw in text)
              for cat, kws in RISK_KEYWORDS.items()}
    best_cat = max(scores, key=scores.get)
    if scores[best_cat] == 0:
        best_cat = '정상'

    neg_words = ['봉쇄', '파업', '급등', '위협', '분쟁', '차질', '불안',
                 '공격', '가뭄', '태풍', '관세', '제재', '인상', '악화']
    pos_words = ['정상화', '완화', '회복', '개설', '협력', '최대', '성장']
    neg_cnt = sum(1 for w in neg_words if w in text)
    pos_cnt = sum(1 for w in pos_words if w in text)
    sentiment = 'negative' if neg_cnt > pos_cnt else (
                'positive' if pos_cnt > neg_cnt else 'neutral')

    return {'category': best_cat, 'risk_weight': RISK_WEIGHTS.get(best_cat, 0),
            'keyword_hits': scores[best_cat], 'sentiment': sentiment}

results = [classify_risk(f"{r['title']} {r['text']}") for _, r in news_df.iterrows()]
news_df['pred_category']  = [r['category']     for r in results]
news_df['risk_weight']    = [r['risk_weight']  for r in results]
news_df['pred_sentiment'] = [r['sentiment']    for r in results]

print('=' * 55)
print('NLP 리스크 분류 결과')
print('=' * 55)
for _, row in news_df.iterrows():
    icon = '🔴' if row['risk_weight'] >= 0.8 else (
           '🟠' if row['risk_weight'] >= 0.6 else (
           '🟡' if row['risk_weight'] >= 0.4 else '🟢'))
    title_short = row['title'][:35] + ('...' if len(row['title']) > 35 else '')
    print(f'  {icon} [{row["pred_category"]:6s}] {title_short}')

print(f'\n카테고리 분포:\n{news_df["pred_category"].value_counts().to_string()}')

## 4. MRI 산출 (Maritime Risk Index)

**MRI 공식**: `0.40 × 부정뉴스비율 + 0.30 × 이벤트빈도 + 0.20 × 운임변동률 + 0.10 × 고위험비율`

운임 변동률은 `data/kcci.csv` 있으면 KCCI 실데이터 사용 (없으면 시뮬).

In [ ]:
def load_kcci():
    """KCCI 운임지수 로드 (실데이터 우선)"""
    if not USE_REAL_KCCI:
        return None
    p = DATA_DIR / 'kcci.csv'
    if not p.exists():
        return None
    try:
        for enc in ['utf-8-sig', 'utf-8', 'cp949']:
            try:
                df = pd.read_csv(p, encoding=enc); break
            except UnicodeDecodeError: continue
        date_col = next((c for c in df.columns
                         if any(k in c for k in ['일자','발표','기준','날짜'])),
                        df.columns[0])
        val_col = next((c for c in df.columns
                        if 'kcci' in c.lower() or '지수' in c or '종합' in c),
                       df.columns[1])
        df = df[[date_col, val_col]].rename(columns={date_col:'date', val_col:'value'})
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df['value'] = pd.to_numeric(df['value'].astype(str).str.replace(',',''),
                                      errors='coerce')
        df = df.dropna().sort_values('date').reset_index(drop=True)
        print(f'✅ KCCI 실데이터 로드: {len(df)}건')
        return df
    except Exception as e:
        print(f'⚠️  KCCI 로드 실패: {e}')
        return None

freight_df = load_kcci()

# ── 시계열 MRI 생성 ──
np.random.seed(42)
dates = pd.date_range('2023-01-01', '2026-04-01', freq='MS')
N = len(dates)

neg_ratio   = np.clip(0.35 + np.random.normal(0, 0.08, N), 0.1, 0.9)
event_cnt_n = np.clip(np.random.poisson(3, N) / 10.0, 0, 1)
high_risk_n = np.clip(np.random.beta(2, 4, N), 0, 1)

if freight_df is not None:
    monthly = freight_df.set_index('date')['value'].resample('MS').mean().reindex(dates, method='nearest')
    chg = monthly.pct_change().fillna(0).abs()
    p95 = np.percentile(chg.values, 95)
    freight_chg_n = np.clip(chg.values / max(p95, 0.01), 0, 1)
else:
    freight_chg_n = np.abs(np.random.normal(0, 0.15, N))

# 홍해 사태 + 미중 관세 패턴
hong_hae = (dates >= '2023-12-01') & (dates <= '2024-06-01')
tariff   = dates >= '2025-04-01'
neg_ratio[hong_hae]     = np.clip(neg_ratio[hong_hae] + 0.3, 0, 1)
event_cnt_n[hong_hae]   = np.clip(event_cnt_n[hong_hae] + 0.25, 0, 1)
freight_chg_n[hong_hae] = np.clip(freight_chg_n[hong_hae] + 0.2, 0, 1)
neg_ratio[tariff]       = np.clip(neg_ratio[tariff] + 0.2, 0, 1)
high_risk_n[tariff]     = np.clip(high_risk_n[tariff] + 0.15, 0, 1)

mri_series = np.clip(
    0.40*neg_ratio + 0.30*event_cnt_n + 0.20*freight_chg_n + 0.10*high_risk_n, 0, 1
)

# ── 오늘 MRI ──
today_neg_ratio = (news_df['pred_sentiment'] == 'negative').mean()
today_event_cnt = min((news_df['pred_category'] != '정상').sum() / 10.0, 1.0)
today_high_risk = (news_df['risk_weight'] >= 0.7).mean()
if freight_df is not None:
    recent_chg = freight_df.tail(8)['value'].pct_change().abs().mean()
    today_freight_chg = min(recent_chg / 0.05, 1.0)
else:
    today_freight_chg = 0.55

today_mri = float(np.clip(
    0.40*today_neg_ratio + 0.30*today_event_cnt
    + 0.20*today_freight_chg + 0.10*today_high_risk, 0, 1))

def mri_grade(m):
    if m >= 0.8: return ('🔴 위험',  '#EF5350')
    if m >= 0.6: return ('🟠 경계',  '#FF7043')
    if m >= 0.3: return ('🟡 주의',  '#FFA726')
    return               ('🟢 정상',  '#66BB6A')

today_grade, today_color = mri_grade(today_mri)
mri_df = pd.DataFrame({'date': dates, 'mri': mri_series})

print('=' * 50)
print('   Maritime Risk Index (MRI) 현황')
print('=' * 50)
print(f'  오늘 MRI: {today_mri:.3f}')
print(f'  등급:    {today_grade}')
print(f'  부정 뉴스 비중: {today_neg_ratio:.0%}')
print(f'  이벤트 감지:   {(news_df["pred_category"]!="정상").sum()}건')

# 시각화
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(mri_df['date'], mri_df['mri'], color='#1F4E79', linewidth=2)
ax.fill_between(mri_df['date'], 0, mri_df['mri'], alpha=0.3, color='#1F4E79')
ax.axhline(0.8, color='#EF5350', linestyle='--', alpha=0.5, label='위험')
ax.axhline(0.6, color='#FF7043', linestyle='--', alpha=0.5, label='경계')
ax.axhline(0.3, color='#FFA726', linestyle='--', alpha=0.5, label='주의')
ax.set_title('Maritime Risk Index (MRI) 시계열 — 2023~2026', fontsize=13, fontweight='bold')
ax.set_ylabel('MRI 점수')
ax.set_ylim(0, 1)
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 부산항 물동량 + 거시경제 변수

부산항 실제 평균 약 200만 TEU/월 (2024년 연간 2,440만 TEU). 실데이터 적용 가능.

In [ ]:
# ── 부산항 물동량 (실데이터 옵션) ──
def load_throughput():
    if not USE_REAL_THROUGHPUT:
        return None
    p = DATA_DIR / 'busan_throughput.csv'
    if not p.exists():
        return None
    try:
        for enc in ['utf-8-sig','utf-8','cp949']:
            try:
                df = pd.read_csv(p, encoding=enc); break
            except UnicodeDecodeError: continue
        date_col = next((c for c in df.columns
                         if any(k in c for k in ['년월','기준','날짜'])), df.columns[0])
        vol_col = next((c for c in df.columns
                        if 'TEU' in c.upper() or '물동량' in c or '처리' in c), df.columns[1])
        df = df[[date_col, vol_col]].rename(columns={date_col:'date', vol_col:'throughput'})
        df['date'] = pd.to_datetime(df['date'].astype(str).str.replace(r'[\.\-/]','',regex=True).str[:6],
                                     format='%Y%m', errors='coerce')
        df['throughput'] = pd.to_numeric(df['throughput'].astype(str).str.replace(',',''), errors='coerce')
        df = df.dropna().sort_values('date').reset_index(drop=True)
        # 단위 자동 변환 → 만 TEU
        avg = df['throughput'].mean()
        if avg > 100000: df['throughput'] /= 10000
        elif avg > 1000: df['throughput'] /= 10
        print(f'✅ 부산항 실데이터: {len(df)}개월, 평균 {df["throughput"].mean():.1f}만 TEU')
        return df
    except Exception as e:
        print(f'⚠️ 물동량 로드 실패: {e}')
        return None

throughput_df = load_throughput()

# 시뮬 또는 실데이터 통합
np.random.seed(42)
if throughput_df is not None:
    dates_m = pd.date_range(throughput_df['date'].min(), throughput_df['date'].max(), freq='MS')
    M = len(dates_m)
    main_df = pd.DataFrame({'date': dates_m}).merge(throughput_df, on='date', how='left')
else:
    dates_m = pd.date_range('2020-01-01', '2026-04-01', freq='MS')
    M = len(dates_m)
    seasonal = np.array([200,185,195,198,205,210,215,220,210,200,205,220]*7)[:M]
    base = seasonal.copy().astype(float)
    base[(dates_m >= '2020-03-01') & (dates_m <= '2021-06-01')] *= 0.88
    base[(dates_m >= '2023-12-01') & (dates_m <= '2024-06-01')] *= 0.94
    base[dates_m >= '2025-04-01'] *= 0.97
    main_df = pd.DataFrame({
        'date': dates_m,
        'throughput': np.clip(base + np.random.normal(0,6,M), 150, 260),
    })

# 거시경제 변수 (시뮬)
main_df['gdp_growth']    = np.random.normal(2.5, 0.8, len(main_df)) / 100
main_df['exchange_rate'] = 1200 + np.cumsum(np.random.normal(0, 15, len(main_df)))
main_df['oil_price']     = 70 + np.cumsum(np.random.normal(0, 2, len(main_df)))
main_df = main_df.ffill().bfill()
main_df['mri'] = np.interp(np.arange(len(main_df)),
                            np.linspace(0, len(main_df)-1, N), mri_series)

print(f'통합 데이터: {len(main_df)}개월')
print(f'  물동량 평균: {main_df["throughput"].mean():.1f}만 TEU/월')
print(f'  환율 평균:   {main_df["exchange_rate"].mean():.0f}원/달러')
main_df.tail()

## 6. LSTM 물동량 예측

**시간순 분할** (data leakage 방지). 앞 80%를 학습, 뒤 20%를 검증.

In [ ]:
if not TORCH_OK:
    print('⚠️ PyTorch 미설치 — LSTM 단계 건너뜀')
    # 다음 단계용 더미
    future_real = main_df['throughput'].tail(3).values
    recent_avg = float(main_df['throughput'].tail(6).mean())
    mape_3m = -1
else:
    FEATURES = ['throughput', 'gdp_growth', 'exchange_rate', 'oil_price', 'mri']
    LOOKBACK = 12
    HORIZON = 3

    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(main_df[FEATURES])

    class TSDataset(Dataset):
        def __init__(self, data, lookback, horizon):
            self.X, self.y = [], []
            for i in range(len(data) - lookback - horizon + 1):
                self.X.append(data[i:i+lookback])
                self.y.append(data[i+lookback:i+lookback+horizon, 0])
            self.X = torch.FloatTensor(np.array(self.X))
            self.y = torch.FloatTensor(np.array(self.y))
        def __len__(self): return len(self.X)
        def __getitem__(self, idx): return self.X[idx], self.y[idx]

    dataset = TSDataset(data_scaled, LOOKBACK, HORIZON)
    train_size = int(len(dataset) * 0.80)
    train_ds = torch.utils.data.Subset(dataset, list(range(train_size)))
    val_ds   = torch.utils.data.Subset(dataset, list(range(train_size, len(dataset))))
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)

    class LSTMModel(nn.Module):
        def __init__(self, in_dim=5, hidden=64, layers=2, horizon=3):
            super().__init__()
            self.lstm = nn.LSTM(in_dim, hidden, layers, dropout=0.2, batch_first=True)
            self.fc   = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(),
                                       nn.Dropout(0.2), nn.Linear(32, horizon))
        def forward(self, x):
            out, _ = self.lstm(x)
            return self.fc(out[:, -1, :])

    torch.manual_seed(42)
    model = LSTMModel(in_dim=len(FEATURES), horizon=HORIZON)
    opt = torch.optim.Adam(model.parameters(), lr=0.005)
    crit = nn.MSELoss()
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=5)

    EPOCHS = 50
    train_losses, val_losses = [], []
    for ep in range(EPOCHS):
        model.train()
        tl = 0
        for X, y in train_loader:
            opt.zero_grad()
            loss = crit(model(X), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += loss.item()
        train_losses.append(tl / len(train_loader))

        model.eval()
        vl = 0
        with torch.no_grad():
            for X, y in val_loader:
                vl += crit(model(X), y).item()
        val_losses.append(vl / max(len(val_loader),1))
        sched.step(val_losses[-1])

    # 예측 & MAPE
    model.eval()
    last_seq = torch.FloatTensor(data_scaled[-LOOKBACK:]).unsqueeze(0)
    with torch.no_grad():
        future_norm = model(last_seq).numpy()[0]

    # 역스케일
    dummy = np.zeros((HORIZON, len(FEATURES)))
    dummy[:, 0] = future_norm
    future_real = scaler.inverse_transform(dummy)[:, 0]

    # 검증 MAPE
    val_pred, val_true = [], []
    with torch.no_grad():
        for X, y in val_loader:
            val_pred.append(model(X).numpy())
            val_true.append(y.numpy())
    val_pred = np.concatenate(val_pred); val_true = np.concatenate(val_true)
    mape_3m = float(mean_absolute_percentage_error(val_true.flatten(), val_pred.flatten()) * 100)

    recent_avg = float(main_df['throughput'].tail(6).mean())
    print(f'\n✅ LSTM 학습 완료 — MAPE: {mape_3m:.2f}%')
    print(f'   3개월 예측: {future_real.round(1).tolist()}')
    print(f'   최근 6개월 평균: {recent_avg:.1f}만 TEU/월')

    # 학습 곡선
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(train_losses, label='Train Loss', color='#1F4E79')
    ax.plot(val_losses, label='Val Loss', color='#D32F2F')
    ax.set_title('LSTM 학습 곡선'); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## 7. 공동선적 매칭 (평상시 운영)

평상시 항로별 그리디 매칭 + 다목적 점수.
**시나리오 발생 시에는 Part 2의 시나리오 정책에 의해 재조정됨.**

In [ ]:
# 공동선적 매칭은 Part 2의 시나리오 운영 재조정과 통합되므로
# 여기서는 기본 데이터 구조만 준비

print('Part 1 완료 — 평상시 운영 흐름 구축됨')
print('  - 뉴스 분류 → MRI → 물동량 예측까지 정상 작동')
print('  - 공동선적 매칭은 Part 2의 시나리오 정책과 통합 실행')
print()
print('Part 2로 진행: 시나리오 기반 운영 재조정 시스템')

---
# Part 2 — 시나리오 기반 운영 재조정 ⭐ NEW

해상 리스크 발생 시 5가지 시나리오에 따라 출하 일정을 다르게 재조정.

| 시나리오 | 조건 | 정책 |
|---|---|---|
| 🟢 **A** 평상시 | MRI < 0.3 | 기존 계획 유지 |
| 🔴 **B** 지정학 분쟁 | MRI ≥ 0.7 + 지정학분쟁 | 우회 + 보류 + 운임 +30% |
| 🟠 **C** 기상 악화 | MRI ≥ 0.5 + 기상재해 | 콜드체인 우선 + 일반 보류 |
| 🟡 **D** 단순 지연 | MRI 0.3~0.5 + 파업/관세 | 집화 일정만 조정 |
| ⚪ **E** 단순 취소 | 주문 cancel | 매칭 그룹 재구성 |

**출력**: 위밋 루티 API 입력용 JSON 표준

## 8. 시나리오 정의 + 자동 분류기

In [ ]:
# 시나리오 5종 — 실제 사례 기반 파라미터
SCENARIOS = {
    'A_NORMAL': {
        'name': '평상시 운영', 'icon': '🟢', 'color': '#4CAF50',
        'trigger': 'MRI < 0.3 + 정상 카테고리',
        'description': '리스크 신호 없음. 기존 계획대로 운송 실행.',
        'delay_days': 0, 'freight_surge_pct': 0.0, 'reroute_required': False,
        'cold_chain_priority': False, 'affects_routes': [],
        'policy': 'AS_PLANNED',
    },
    'B_GEOPOLITICAL': {
        'name': '지정학 분쟁 (호르무즈/홍해)', 'icon': '🔴', 'color': '#D32F2F',
        'trigger': 'MRI ≥ 0.7 + 지정학분쟁',
        'description': '항로 봉쇄·전쟁 위험. 케이프타운 우회 + 운임 +30% (실제 홍해 사태 사례).',
        'delay_days': 14,            # 케이프타운 우회 약 +14일
        'freight_surge_pct': 0.30,   # 홍해 사태 시 실제 +30%
        'reroute_required': True,
        'cold_chain_priority': True,
        'affects_routes': ['부산→로테르담'],  # 수에즈 경유 항로만
        'policy': 'REROUTE_AND_HOLDBACK',
    },
    'C_WEATHER': {
        'name': '기상 악화 (태풍/폭풍)', 'icon': '🟠', 'color': '#F57C00',
        'trigger': 'MRI ≥ 0.5 + 기상재해',
        'description': '단기 출항 지연 5일. 콜드체인 우선 처리, 일반화물 보류.',
        'delay_days': 5, 'freight_surge_pct': 0.05, 'reroute_required': False,
        'cold_chain_priority': True, 'affects_routes': [],
        'policy': 'HOLDBACK_NORMAL_RUSH_COLD',
    },
    'D_DELAY': {
        'name': '단순 출항 지연', 'icon': '🟡', 'color': '#FBC02D',
        'trigger': 'MRI 0.3~0.5 + 파업/관세/운임급등',
        'description': '항만 혼잡·파업·운임 변동 등 일반 지연 3일. 집화 일정 조정.',
        'delay_days': 3, 'freight_surge_pct': 0.02, 'reroute_required': False,
        'cold_chain_priority': False, 'affects_routes': [],
        'policy': 'SHIFT_PICKUP',
    },
    'E_CANCELLATION': {
        'name': '고객 단순 변심 (주문 취소)', 'icon': '⚪', 'color': '#9E9E9E',
        'trigger': '특정 주문 cancel_flag = True',
        'description': '단일 화주 출하 취소. 잔여 화물로 매칭 그룹 재구성.',
        'delay_days': 0, 'freight_surge_pct': 0.0, 'reroute_required': False,
        'cold_chain_priority': False, 'affects_routes': [],
        'policy': 'REGROUP_REMAINING',
    },
}

# ── 자동 시나리오 분류기 (MRI + NLP 결과 기반) ──
def auto_classify_scenario(today_mri: float, top_category: str,
                            cancel_count: int = 0) -> str:
    """
    Part 1의 MRI 점수 + NLP 카테고리로부터 시나리오 자동 추론

    우선순위 규칙:
    1. cancel_count > 0 → E_CANCELLATION
    2. 카테고리 + MRI 매칭 (지정학 → 기상 → 파업/관세/운임급등)
    3. 카테고리 불명확 시 MRI 점수만으로 fallback
    """
    if cancel_count > 0:
        return 'E_CANCELLATION'

    # 카테고리 + MRI 매칭 (구체적 → 일반적 순서)
    if top_category == '지정학분쟁' and today_mri >= 0.7:
        return 'B_GEOPOLITICAL'
    if top_category == '기상재해' and today_mri >= 0.5:
        return 'C_WEATHER'
    # 운임급등도 D 시나리오로 (단순 지연 카테고리)
    if top_category in ['항만파업', '관세정책', '운임급등'] and today_mri >= 0.3:
        return 'D_DELAY'
    if today_mri < 0.3:
        return 'A_NORMAL'

    # 카테고리 불명확 시 MRI로 fallback
    if today_mri >= 0.7: return 'B_GEOPOLITICAL'
    if today_mri >= 0.5: return 'C_WEATHER'
    if today_mri >= 0.3: return 'D_DELAY'
    return 'A_NORMAL'

# 자동 분류 결과
top_category = news_df['pred_category'].mode()[0] if len(news_df) > 0 else '정상'
auto_scenario_id = auto_classify_scenario(today_mri, top_category, cancel_count=0)

print('=' * 60)
print('자동 시나리오 분류 결과')
print('=' * 60)
print(f'  오늘 MRI:       {today_mri:.3f}')
print(f'  최다 카테고리:  {top_category}')
print(f'  → 자동 분류:   {auto_scenario_id} ({SCENARIOS[auto_scenario_id]["icon"]} {SCENARIOS[auto_scenario_id]["name"]})')
print()
print('▶ 사용자가 수동 오버라이드 가능 (Part 3에서 5개 시나리오 일괄 비교)')

## 9. 출하 예정 건 등록 (시뮬)

평상시 사전 등록되는 중소 수출기업의 출하 예정 건. 30건으로 충분한 매칭 가능.

In [ ]:
# ── 항로 정보 (실제 부산항 운임 기준) ──
ROUTE_INFO = {
    '부산→상하이':  {'distance_nm': 485,   'usd_per_teu': 950,  'transit_days': 3},
    '부산→도쿄':    {'distance_nm': 610,   'usd_per_teu': 680,  'transit_days': 2},
    '부산→LA':      {'distance_nm': 5040,  'usd_per_teu': 2300, 'transit_days': 14},
    '부산→로테르담': {'distance_nm': 11200, 'usd_per_teu': 3500, 'transit_days': 28},
    '부산→싱가포르': {'distance_nm': 2650,  'usd_per_teu': 1050, 'transit_days': 7},
}
LCL_MULTIPLIER = 1.5  # LCL은 FCL 대비 약 1.5배 단가

def calc_freight(cbm: float, route: str) -> int:
    info = ROUTE_INFO[route]
    fcl_per_cbm = info['usd_per_teu'] / 33  # 1 TEU ≈ 33 CBM
    return round(fcl_per_cbm * LCL_MULTIPLIER * cbm)

# ── 출하 예정 건 시뮬 (30건) ──
np.random.seed(42)
base_date = datetime(2026, 5, 1)
COMPANIES = [f"화주_{chr(65+i)}" for i in range(30)]
REGIONS = ['경기남부', '경기북부', '충청', '경상남부', '경상북부']
CARGO_TYPES = ['일반화물', '냉장화물', '위험물']

shipments = []
for i, company in enumerate(COMPANIES):
    route = np.random.choice(list(ROUTE_INFO.keys()))
    cargo_type = np.random.choice(CARGO_TYPES, p=[0.65, 0.25, 0.10])
    region = np.random.choice(REGIONS)
    pickup_date = base_date + timedelta(days=int(np.random.randint(2, 16)))
    cbm = round(np.random.uniform(5, 35), 1)
    deadline_days = int(np.random.choice([7, 10, 14, 21], p=[0.2, 0.3, 0.3, 0.2]))
    urgent = np.random.random() < 0.15
    shipments.append({
        'shipment_id':    f'SH-{i+1:03d}',
        'company':        company,
        'route':          route,
        'cargo_type':     cargo_type,
        'region':         region,
        'pickup_date':    pickup_date,
        'cbm':            cbm,
        'deadline_days':  deadline_days,
        'urgent':         urgent,
        'estimated_cost': calc_freight(cbm, route),
    })

ship_df = pd.DataFrame(shipments)

print('=' * 60)
print(f'출하 예정 건 등록: {len(ship_df)}건')
print('=' * 60)
print(f'  화물 유형: {dict(ship_df["cargo_type"].value_counts())}')
print(f'  항로 분포: {dict(ship_df["route"].value_counts())}')
print(f'  권역 분포: {dict(ship_df["region"].value_counts())}')
print(f'  긴급 화물: {ship_df["urgent"].sum()}건')
print(f'  총 예상 운임: ${ship_df["estimated_cost"].sum():,}')

ship_df.head(10)

## 10. 영향 분석 엔진

각 출하 건에 시나리오를 적용해 영향(지연·비용·납기 위반)을 분석.

In [ ]:
@dataclass
class ImpactAnalysis:
    """시나리오 적용 후 출하 건별 영향 분석 결과"""
    shipment_id:        str
    is_affected:        bool
    delay_days_applied: int
    new_pickup_date:    datetime
    new_estimated_cost: int
    cost_delta:         int
    deadline_violated:  bool
    requires_holdback:  bool
    requires_priority:  bool
    reason:             str

def _as_planned(shipment: dict) -> ImpactAnalysis:
    return ImpactAnalysis(
        shipment_id=shipment['shipment_id'], is_affected=False,
        delay_days_applied=0, new_pickup_date=shipment['pickup_date'],
        new_estimated_cost=shipment['estimated_cost'], cost_delta=0,
        deadline_violated=False, requires_holdback=False,
        requires_priority=False, reason='평상시 — 기존 계획 유지'
    )

def analyze_impact(shipment: dict, scenario: dict,
                    cancelled_ids: set = None) -> ImpactAnalysis:
    cancelled_ids = cancelled_ids or set()

    # E 시나리오: 취소 처리
    if scenario['policy'] == 'REGROUP_REMAINING':
        if shipment['shipment_id'] in cancelled_ids:
            return ImpactAnalysis(
                shipment_id=shipment['shipment_id'], is_affected=True,
                delay_days_applied=0, new_pickup_date=shipment['pickup_date'],
                new_estimated_cost=0,
                cost_delta=-shipment['estimated_cost'],
                deadline_violated=False, requires_holdback=False,
                requires_priority=False,
                reason='주문 취소 — 매칭 그룹에서 제외',
            )
        return _as_planned(shipment)

    # A 시나리오: 평상시
    if scenario['policy'] == 'AS_PLANNED':
        return _as_planned(shipment)

    # 항로 필터 (B는 특정 항로만)
    if scenario['affects_routes'] and shipment['route'] not in scenario['affects_routes']:
        return ImpactAnalysis(
            shipment_id=shipment['shipment_id'], is_affected=False,
            delay_days_applied=0, new_pickup_date=shipment['pickup_date'],
            new_estimated_cost=shipment['estimated_cost'], cost_delta=0,
            deadline_violated=False, requires_holdback=False,
            requires_priority=False,
            reason=f'영향권 외 항로 ({shipment["route"]})'
        )

    # 영향 적용
    delay = scenario['delay_days']
    surge = scenario['freight_surge_pct']
    new_pickup = shipment['pickup_date'] + timedelta(days=delay)
    new_cost = round(shipment['estimated_cost'] * (1 + surge))
    cost_delta = new_cost - shipment['estimated_cost']

    deadline_violated = delay > shipment['deadline_days']
    requires_holdback = delay >= 3 and not shipment['urgent']
    requires_priority = (
        scenario['cold_chain_priority']
        and shipment['cargo_type'] == '냉장화물'
    ) or shipment['urgent']

    reason = f'{scenario["name"]} — 지연 {delay}일'
    if surge > 0: reason += f', 운임 +{surge:.0%}'
    if scenario['reroute_required']: reason += ', 우회항로'

    return ImpactAnalysis(
        shipment_id=shipment['shipment_id'], is_affected=True,
        delay_days_applied=delay, new_pickup_date=new_pickup,
        new_estimated_cost=new_cost, cost_delta=cost_delta,
        deadline_violated=deadline_violated,
        requires_holdback=requires_holdback,
        requires_priority=requires_priority,
        reason=reason,
    )

# 자동 분류된 시나리오로 영향 분석
auto_scenario = SCENARIOS[auto_scenario_id]
impacts_auto = [analyze_impact(s, auto_scenario) for s in ship_df.to_dict('records')]

affected = sum(1 for ia in impacts_auto if ia.is_affected)
violated = sum(1 for ia in impacts_auto if ia.deadline_violated)
total_delta = sum(ia.cost_delta for ia in impacts_auto)

print('=' * 60)
print(f'영향 분석 결과 — {auto_scenario["icon"]} {auto_scenario["name"]}')
print('=' * 60)
print(f'  영향 받는 출하: {affected} / {len(ship_df)}건')
print(f'  납기 위반 위험: {violated}건')
print(f'  총 비용 영향:   ${total_delta:+,}')
print()
print('▶ 영향 받는 출하 건 (상위 5건):')
for ia in [x for x in impacts_auto if x.is_affected][:5]:
    print(f'  {ia.shipment_id}: 지연 {ia.delay_days_applied}일, '
          f'운임 ${ia.cost_delta:+,}, 납기위반={ia.deadline_violated}')

## 11. 운영 재조정 엔진

영향 분석 결과를 바탕으로 4가지 행동을 결정:
1. **우선처리(PRIORITY)**: 콜드체인·긴급 화물
2. **반입 보류(HOLDBACK)**: 출항 지연 시 항만 반입 미루기 → 보관료 절감
3. **집화 이동(SHIFT)**: 단순 일정 조정
4. **권역 통합(CONSOLIDATION)**: 같은 권역 + 비슷한 새 집화일 → 통합 집화

In [ ]:
def reorganize_pickups(ship_df: pd.DataFrame, impacts: list, scenario: dict) -> dict:
    """
    영향 분석 → 4가지 행동으로 분류 + 권역 통합 그룹 구성
    """
    impact_map = {ia.shipment_id: ia for ia in impacts}

    pickup_holdback = []
    pickup_shifted  = []
    pickup_priority = []

    for _, ship in ship_df.iterrows():
        ia = impact_map[ship['shipment_id']]
        if not ia.is_affected:
            continue
        if ia.requires_priority:
            pickup_priority.append({
                'shipment_id': ship['shipment_id'],
                'company':     ship['company'],
                'cargo_type':  ship['cargo_type'],
                'urgency_reason': '냉장화물' if ship['cargo_type']=='냉장화물' else '긴급',
                'new_pickup_date': ia.new_pickup_date.strftime('%Y-%m-%d'),
            })
        elif ia.requires_holdback:
            pickup_holdback.append({
                'shipment_id': ship['shipment_id'],
                'company':     ship['company'],
                'original_pickup': ship['pickup_date'].strftime('%Y-%m-%d'),
                'reason': f'출항 {ia.delay_days_applied}일 지연 — 항만 반입 보류',
            })
        else:
            pickup_shifted.append({
                'shipment_id': ship['shipment_id'],
                'company':     ship['company'],
                'original_pickup': ship['pickup_date'].strftime('%Y-%m-%d'),
                'new_pickup':  ia.new_pickup_date.strftime('%Y-%m-%d'),
                'shift_days':  ia.delay_days_applied,
            })

    # 권역별 통합 집화 그룹 (지연 일정 조정 대상 중)
    shifted_ids = {p['shipment_id'] for p in pickup_shifted}
    candidates = ship_df[ship_df['shipment_id'].isin(shifted_ids)].copy()
    if len(candidates) > 0:
        candidates['new_pickup'] = candidates['shipment_id'].apply(
            lambda sid: impact_map[sid].new_pickup_date
        )

    consolidation_groups = []
    used = set()
    CARGO_COMPAT = {('일반화물','일반화물'): True, ('냉장화물','냉장화물'): True,
                    ('위험물','위험물'): True}
    def compat(c1, c2):
        return CARGO_COMPAT.get((c1,c2), CARGO_COMPAT.get((c2,c1), False))

    if len(candidates) > 0:
        for region in candidates['region'].unique():
            region_df = candidates[candidates['region'] == region].sort_values('new_pickup')
            for _, row in region_df.iterrows():
                if row['shipment_id'] in used: continue
                group = [row.to_dict()]
                used.add(row['shipment_id'])
                for _, row2 in region_df.iterrows():
                    if row2['shipment_id'] in used: continue
                    date_diff = abs((row2['new_pickup'] - row['new_pickup']).days)
                    if date_diff <= 2 and compat(row['cargo_type'], row2['cargo_type']):
                        group.append(row2.to_dict())
                        used.add(row2['shipment_id'])
                if len(group) >= 2:
                    consolidation_groups.append({
                        'region':     region,
                        'cargo_type': row['cargo_type'],
                        'merged_pickup_date': row['new_pickup'].strftime('%Y-%m-%d'),
                        'members':    [g['shipment_id'] for g in group],
                        'companies':  [g['company'] for g in group],
                        'total_cbm':  round(sum(g['cbm'] for g in group), 1),
                        'savings_estimate_pct': 0.15,  # 권역 통합 약 15% 절감
                    })

    return {
        'pickup_holdback':      pickup_holdback,
        'pickup_shifted':       pickup_shifted,
        'pickup_priority':      pickup_priority,
        'consolidation_groups': consolidation_groups,
    }

# 자동 시나리오 적용
reorg_auto = reorganize_pickups(ship_df, impacts_auto, auto_scenario)

print('=' * 60)
print(f'운영 재조정 — {auto_scenario["icon"]} {auto_scenario["name"]}')
print('=' * 60)
print(f'  ★ 우선처리:    {len(reorg_auto["pickup_priority"])}건')
print(f'  ◐ 반입보류:    {len(reorg_auto["pickup_holdback"])}건')
print(f'  → 집화이동:    {len(reorg_auto["pickup_shifted"])}건')
print(f'  ◆ 권역통합:    {len(reorg_auto["consolidation_groups"])}그룹')

if reorg_auto['consolidation_groups']:
    print('\n▶ 권역 통합 그룹 (상위 3개):')
    for g in reorg_auto['consolidation_groups'][:3]:
        print(f'  [{g["region"]}/{g["cargo_type"]}] {", ".join(g["companies"])} '
              f'({g["total_cbm"]}CBM, {g["merged_pickup_date"]} 집화)')

## 12. 루티 연계 JSON 출력 ⭐ 핵심

위밋 루티/루티프로 API에 직접 전달 가능한 표준 JSON.

> **현재**: 시뮬레이션 모드 (integration_status='simulation_mode')
> **향후**: 루티 API 발급 시 즉시 'live_api' 모드 전환 가능

In [ ]:
def generate_routy_input(scenario_id: str, scenario: dict, ship_df: pd.DataFrame,
                          impacts: list, reorg: dict) -> dict:
    """루티 API 입력용 표준 JSON 생성"""
    impact_map = {ia.shipment_id: ia for ia in impacts}

    pickup_adjustments = []
    for _, s in ship_df.iterrows():
        ia = impact_map[s['shipment_id']]
        if not ia.is_affected: continue
        pickup_adjustments.append({
            'shipment_id':       s['shipment_id'],
            'company':           s['company'],
            'region':            s['region'],
            'cbm':               float(s['cbm']),
            'cargo_type':        s['cargo_type'],
            'route':             s['route'],
            'original_pickup':   s['pickup_date'].strftime('%Y-%m-%d'),
            'adjusted_pickup':   ia.new_pickup_date.strftime('%Y-%m-%d'),
            'action':            'PRIORITY' if ia.requires_priority else (
                                  'HOLDBACK' if ia.requires_holdback else 'SHIFT'),
            'cost_delta_usd':    int(ia.cost_delta),
            'deadline_violated': bool(ia.deadline_violated),
            'cold_chain':        s['cargo_type'] == '냉장화물',
        })

    return {
        'execution_group_id': f'EG-{datetime.today().strftime("%Y%m%d")}-{scenario_id}',
        'generated_at':       datetime.today().isoformat(timespec='seconds'),
        'scenario': {
            'id':          scenario_id,
            'name':        scenario['name'],
            'icon':        scenario['icon'],
            'policy':      scenario['policy'],
            'description': scenario['description'],
        },
        'summary': {
            'total_shipments':       int(len(ship_df)),
            'affected':              int(sum(1 for ia in impacts if ia.is_affected)),
            'priority':              int(len(reorg['pickup_priority'])),
            'holdback':              int(len(reorg['pickup_holdback'])),
            'shifted':               int(len(reorg['pickup_shifted'])),
            'consolidation_groups':  int(len(reorg['consolidation_groups'])),
            'total_cost_delta_usd':  int(sum(ia.cost_delta for ia in impacts)),
            'deadline_violations':   int(sum(1 for ia in impacts if ia.deadline_violated)),
        },
        'pickup_adjustments':   pickup_adjustments,
        'consolidation_groups': reorg['consolidation_groups'],
        'priority_routing':     [p['shipment_id'] for p in reorg['pickup_priority']],
        'holdback_list':        [p['shipment_id'] for p in reorg['pickup_holdback']],
        'cargo_special_handling': {
            'cold_chain_count': int(sum(1 for s in ship_df.to_dict('records')
                                         if s['cargo_type']=='냉장화물')),
            'hazardous_count':  int(sum(1 for s in ship_df.to_dict('records')
                                         if s['cargo_type']=='위험물')),
        },
        'meta': {
            'note':               '본 출력은 위밋 루티/루티프로 API 입력 스펙으로 설계됨',
            'integration_status': 'simulation_mode',  # 실서비스 시 'live_api'
            'next_step_api':      'POST /v1/dispatch/execute',
        }
    }

# 자동 시나리오로 JSON 생성
routy_input_auto = generate_routy_input(
    auto_scenario_id, auto_scenario, ship_df, impacts_auto, reorg_auto
)

# JSON 미리보기
preview = {
    'execution_group_id': routy_input_auto['execution_group_id'],
    'scenario':           routy_input_auto['scenario'],
    'summary':            routy_input_auto['summary'],
    'pickup_adjustments[0]': (routy_input_auto['pickup_adjustments'][:1]
                                 if routy_input_auto['pickup_adjustments'] else []),
    'consolidation_groups[0]': (routy_input_auto['consolidation_groups'][:1]
                                 if routy_input_auto['consolidation_groups'] else []),
    'meta':               routy_input_auto['meta'],
}
print('=' * 60)
print('루티 API 입력 JSON 미리보기')
print('=' * 60)
print(json.dumps(preview, ensure_ascii=False, indent=2, default=str))

# 파일 저장
out_path = Path('./routy_inputs')
out_path.mkdir(exist_ok=True)
fp = out_path / f'{routy_input_auto["execution_group_id"]}.json'
with open(fp, 'w', encoding='utf-8') as f:
    json.dump(routy_input_auto, f, ensure_ascii=False, indent=2, default=str)
print(f'\n✅ JSON 저장: {fp}')

---
# Part 3 — 통합 시연

5개 시나리오를 모두 실행하여 결과를 비교. 발표장에서 사용자 선택 시연 가능.

## 13. 5개 시나리오 일괄 실행 + 비교

In [ ]:
# 5개 시나리오 모두 실행
# E 시나리오용 가상 취소 ID
cancelled_ids_demo = {'SH-001', 'SH-005', 'SH-009'}

results = {}
for scenario_id, scenario in SCENARIOS.items():
    impacts = [analyze_impact(s, scenario, cancelled_ids_demo)
               for s in ship_df.to_dict('records')]
    reorg = reorganize_pickups(ship_df, impacts, scenario)
    routy_input = generate_routy_input(scenario_id, scenario, ship_df, impacts, reorg)
    results[scenario_id] = {
        'scenario': scenario, 'impacts': impacts,
        'reorg': reorg, 'routy_input': routy_input,
    }
    # JSON 파일 저장
    fp = out_path / f'{routy_input["execution_group_id"]}.json'
    with open(fp, 'w', encoding='utf-8') as f:
        json.dump(routy_input, f, ensure_ascii=False, indent=2, default=str)

# ── 비교 표 ──
print('=' * 80)
print('               5개 시나리오 비교 (출하 30건 기준)')
print('=' * 80)
print(f'{"시나리오":<28s} {"영향":>5s} {"우선":>5s} {"보류":>5s} {"이동":>5s} {"통합":>5s} {"비용Δ":>10s}')
print('─' * 80)
for sid, res in results.items():
    s = res['routy_input']['summary']
    sc = res['scenario']
    name = f'{sc["icon"]} {sid.split("_")[0]} {sc["name"][:18]}'
    print(f'{name:<28s} {s["affected"]:>5d} {s["priority"]:>5d} '
          f'{s["holdback"]:>5d} {s["shifted"]:>5d} '
          f'{s["consolidation_groups"]:>5d} ${s["total_cost_delta_usd"]:>+9,d}')

print()
print('▶ 시나리오별 정책 요약:')
for sid, res in results.items():
    sc = res['scenario']
    print(f'  {sc["icon"]} {sid}: {sc["description"]}')

## 14. 핵심 KPI 시각화

In [ ]:
# 시나리오별 KPI 시각화
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.patch.set_facecolor('white')

scenario_ids = list(SCENARIOS.keys())
labels = [f'{SCENARIOS[sid]["icon"]} {sid.split("_")[0]}' for sid in scenario_ids]
colors = [SCENARIOS[sid]['color'] for sid in scenario_ids]

# (1) 영향 받는 출하 건수
ax1 = axes[0, 0]
affected_counts = [results[sid]['routy_input']['summary']['affected'] for sid in scenario_ids]
bars = ax1.bar(labels, affected_counts, color=colors, edgecolor='black')
ax1.set_title('시나리오별 영향 받는 출하 건수', fontsize=12, fontweight='bold')
ax1.set_ylabel('건수')
ax1.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, affected_counts):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
              f'{val}', ha='center', fontweight='bold')

# (2) 행동 분류 stacked bar
ax2 = axes[0, 1]
priority_v = [results[sid]['routy_input']['summary']['priority'] for sid in scenario_ids]
holdback_v = [results[sid]['routy_input']['summary']['holdback'] for sid in scenario_ids]
shifted_v  = [results[sid]['routy_input']['summary']['shifted']  for sid in scenario_ids]
ax2.bar(labels, priority_v, label='우선처리', color='#D32F2F', edgecolor='black')
ax2.bar(labels, holdback_v, bottom=priority_v, label='반입보류',
         color='#F57C00', edgecolor='black')
ax2.bar(labels, shifted_v,
         bottom=[p+h for p,h in zip(priority_v, holdback_v)],
         label='집화이동', color='#2E75B6', edgecolor='black')
ax2.set_title('시나리오별 행동 분류 (Priority/Holdback/Shift)', fontsize=12, fontweight='bold')
ax2.set_ylabel('건수')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# (3) 비용 영향 (절댓값)
ax3 = axes[1, 0]
cost_deltas = [results[sid]['routy_input']['summary']['total_cost_delta_usd']
               for sid in scenario_ids]
bar_colors = ['#4CAF50' if c < 0 else ('#FFA726' if c == 0 else '#D32F2F')
              for c in cost_deltas]
bars = ax3.bar(labels, cost_deltas, color=bar_colors, edgecolor='black')
ax3.axhline(0, color='black', linewidth=0.8)
ax3.set_title('시나리오별 운임 비용 변화', fontsize=12, fontweight='bold')
ax3.set_ylabel('USD')
ax3.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, cost_deltas):
    y_offset = 50 if val >= 0 else -50
    va = 'bottom' if val >= 0 else 'top'
    ax3.text(bar.get_x()+bar.get_width()/2, val+y_offset,
              f'${val:+,}', ha='center', va=va, fontweight='bold', fontsize=9)

# (4) 납기 위반 위험
ax4 = axes[1, 1]
violations = [results[sid]['routy_input']['summary']['deadline_violations']
              for sid in scenario_ids]
bars = ax4.bar(labels, violations, color=colors, edgecolor='black')
ax4.set_title('시나리오별 납기 위반 위험 건수', fontsize=12, fontweight='bold')
ax4.set_ylabel('건수')
ax4.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, violations):
    if val > 0:
        ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                  f'{val}', ha='center', fontweight='bold', color='red')

plt.suptitle('해상 리스크 시나리오 대응 KPI 대시보드',
              fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 15. 발표용 케이스 스터디

**시나리오**: 경기남부 권역의 화주 A, B, C가 콜드체인 식품을 부산항 반입 후 미국행 선적 예정.
호르무즈 해협 봉쇄가 발생했을 때 본 플랫폼이 어떻게 대응하는지 시연.

In [ ]:
# 케이스 스터디용 가상 화주 데이터 추가
case_study_shipments = pd.DataFrame([
    {'shipment_id': 'CASE-A', 'company': '경기A사 (식품제조)',
     'route': '부산→LA', 'cargo_type': '냉장화물', 'region': '경기남부',
     'pickup_date': datetime(2026,5,12), 'cbm': 8.0, 'deadline_days': 14,
     'urgent': False, 'estimated_cost': calc_freight(8.0, '부산→LA')},
    {'shipment_id': 'CASE-B', 'company': '경기B사 (가공식품)',
     'route': '부산→LA', 'cargo_type': '냉장화물', 'region': '경기남부',
     'pickup_date': datetime(2026,5,13), 'cbm': 6.5, 'deadline_days': 10,
     'urgent': False, 'estimated_cost': calc_freight(6.5, '부산→LA')},
    {'shipment_id': 'CASE-C', 'company': '경기C사 (해산물)',
     'route': '부산→LA', 'cargo_type': '냉장화물', 'region': '경기남부',
     'pickup_date': datetime(2026,5,14), 'cbm': 9.2, 'deadline_days': 14,
     'urgent': True, 'estimated_cost': calc_freight(9.2, '부산→LA')},
])

print('=' * 70)
print('케이스 스터디: 경기남부 콜드체인 식품 화주 A/B/C')
print('=' * 70)
print(case_study_shipments[['shipment_id','company','cargo_type','cbm',
                              'pickup_date','estimated_cost']].to_string(index=False))

# 평상시 (시나리오 A) 비교
case_a_impacts = [analyze_impact(s, SCENARIOS['A_NORMAL'])
                   for s in case_study_shipments.to_dict('records')]
case_a_reorg = reorganize_pickups(case_study_shipments, case_a_impacts, SCENARIOS['A_NORMAL'])

# 호르무즈 봉쇄 (시나리오 B)
# 다만 '부산→LA'는 affects_routes=['부산→로테르담']에 안 잡히므로
# 케이스 스터디용으로 시나리오 B를 미국향까지 확장한 가상 시나리오 생성
scenario_b_extended = SCENARIOS['B_GEOPOLITICAL'].copy()
scenario_b_extended['affects_routes'] = ['부산→로테르담', '부산→LA']  # 데모용

case_b_impacts = [analyze_impact(s, scenario_b_extended)
                   for s in case_study_shipments.to_dict('records')]
case_b_reorg = reorganize_pickups(case_study_shipments, case_b_impacts, scenario_b_extended)

print('\n--- 평상시 (시나리오 A) ---')
print(f'  영향:      {sum(1 for ia in case_a_impacts if ia.is_affected)}건')
print(f'  통합 그룹: {len(case_a_reorg["consolidation_groups"])}개')
print(f'  비용 변화: ${sum(ia.cost_delta for ia in case_a_impacts):+,}')

print('\n--- 호르무즈 봉쇄 (시나리오 B 확장) ---')
print(f'  영향:      {sum(1 for ia in case_b_impacts if ia.is_affected)}건')
print(f'  우선처리:  {len(case_b_reorg["pickup_priority"])}건 (콜드체인)')
print(f'  보류:      {len(case_b_reorg["pickup_holdback"])}건')
print(f'  비용 변화: ${sum(ia.cost_delta for ia in case_b_impacts):+,}')

for ia in case_b_impacts:
    print(f'\n  {ia.shipment_id}: {ia.reason}')
    print(f'     원래 집화 → 새 집화: '
          f'{[s for s in case_study_shipments.to_dict("records") if s["shipment_id"]==ia.shipment_id][0]["pickup_date"].strftime("%m-%d")}'
          f' → {ia.new_pickup_date.strftime("%m-%d")}')
    print(f'     운임: ${ia.cost_delta:+,}, 납기위반: {ia.deadline_violated}, '
          f'우선처리: {ia.requires_priority}, 보류: {ia.requires_holdback}')

# 발표용 멘트
print('\n' + '=' * 70)
print('발표 시 사용 멘트 (60초)')
print('=' * 70)
print("""
"호르무즈 해협 봉쇄 시나리오를 가정해보겠습니다.

평상시에는 경기남부 A/B/C 3개사의 콜드체인 식품이 각각 5월 12, 13, 14일에
개별 집화되어 부산항으로 이동합니다.

그런데 호르무즈 봉쇄가 발생하면, 본 플랫폼은:

(1) 우선처리 — A/B/C 모두 콜드체인이라 자동으로 우선처리 대상으로 분류
(2) 보류 분석 — 출항 14일 지연이 예상되므로 항만 반입을 미루고
                  내륙 보관 비용 절감 전략 제시
(3) 운임 대응 — 운임 +30% 상승분을 사전 경보, 대체 항로 검토 권고
(4) 루티 연계 — 조정된 집화 일정과 우선순위를 JSON으로 자동 출력하여
                  위밋 루티/루티프로가 즉시 배차 최적화에 사용 가능

이 모든 과정이 자동으로, 화주가 개별 대응할 필요 없이, 권역 단위로
재조정됩니다. 이게 본 플랫폼의 핵심 가치입니다."
""")

---
## 결론

### 본 플랫폼의 핵심 가치

1. **선제적 의사결정 자동화**: MRI 리스크 신호 발생 시 자동으로 시나리오 분류 → 영향 분석 → 운영 재조정까지 체인으로 처리
2. **시나리오별 차별화 대응**: 같은 "지연"이라도 원인에 따라 정책이 다름 (지정학 vs 기상 vs 단순지연 vs 취소)
3. **위밋 루티 연계용 JSON 표준**: 향후 루티 API 발급 시 즉시 연동 가능한 구조
4. **권역 단위 운영 재조정**: 개별 화주 대응이 아닌 권역 단위 통합 집화로 협상력·효율성 확보

### 향후 확장

- **루티 API 실연동**: 시뮬레이션 → 실시간 배차 최적화로 전환
- **AIS 실시간 선박 위치**: MarineTraffic API 연동으로 시나리오 자동 트리거
- **KoBERT 임베딩**: 키워드 사전 → 한국어 BERT fine-tuning
- **CBAM 탄소 데이터**: ESG 모듈 추가 (v3 셀 43 이식)
- **Hungarian 최적 매칭**: 화주 수 증가 시 그리디 → 최적 매칭으로 확장